# 007 Event Streaming

这是 LangGraph 学习线的第七份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/event-streaming

学习目标：

1. 理解 event streaming 为什么是 agent UI 和服务端观察性的核心能力
2. 学会使用 `.stream(..., stream_mode="updates")`
3. 学会使用 `.stream(..., stream_mode="values")`
4. 学会使用 `.stream(..., stream_mode="debug")`
5. 了解 `astream_events(...)` 的事件结构，并对比本仓库 Harness SSE
6. 理解官方新版 `stream_events(..., version="v3")` 的 typed projections 设计

官方页面使用较新的 event API。当前环境是 `langgraph==1.1.9`，本课使用当前可运行的 streaming API，并补充说明和新版 event API 的关系。

## 1. Event streaming 解决什么问题

如果 agent 运行时间很短，直接等最终结果就可以。

但真实 agent 往往会：

- 调多个工具
- 等待审批
- 进入多个节点
- 产生中间观察
- 需要在前端实时展示进度

这时就需要 event streaming。

核心目标：

```text
不是只看最终 answer，
而是观察 graph 每一步发生了什么。
```

## 2. 官方文档里的 Event Streaming 层

官方页面把 LangGraph streaming 分成两层：

| 层级 | 作用 | 典型 API |
| --- | --- | --- |
| 底层 streaming | 直接观察 graph 执行事件 | `.stream(..., stream_mode="updates")` |
| event streaming | 把底层事件转换成更好消费的 typed projections | `.stream_events(..., version="v3")` |

新版 event streaming 的重点不是换一个名字，而是让应用代码可以同时消费多个投影：

- `stream.messages`：模型消息和 token 增量
- `stream.values`：完整 state 快照
- `stream.output`：最终输出
- `stream.subgraphs`：子图执行
- `stream.interrupts`：人工审批或人工输入中断
- `stream.extensions`：自定义 transformer 产生的业务投影

当前仓库环境是 `langgraph==1.1.9`，还没有同步 `stream_events` 和官方页面里的 `langgraph.stream` transformer API。所以下面的代码优先演示当前可运行的底层 streaming，并把新版 event streaming 当作设计对照来理解。

In [88]:
import asyncio
import importlib.metadata
import json
import threading
from typing import Any

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

langgraph 1.2.1


## 3. 定义一个三节点 graph

这个 graph 很简单：

```text
START -> collect_notes -> draft_answer -> finalize -> END
```

我们用它观察不同 stream mode 的输出差异。

In [89]:
class EventState(TypedDict):
    topic: str
    notes: list[str]
    draft: str
    final: str


def collect_notes(state: EventState) -> dict:
    return {
        "notes": [
            state["topic"] + " 需要观察节点更新",
            state["topic"] + " 需要区分 updates 和 values",
        ]
    }


def draft_answer(state: EventState) -> dict:
    return {
        "draft": "草稿：" + "；".join(state["notes"]),
    }


def finalize(state: EventState) -> dict:
    return {
        "final": "最终回答：" + state["draft"],
    }


event_graph_builder = StateGraph(EventState)
event_graph_builder.add_node("collect_notes", collect_notes)
event_graph_builder.add_node("draft_answer", draft_answer)
event_graph_builder.add_node("finalize", finalize)

event_graph_builder.add_edge(START, "collect_notes")
event_graph_builder.add_edge("collect_notes", "draft_answer")
event_graph_builder.add_edge("draft_answer", "finalize")
event_graph_builder.add_edge("finalize", END)

event_graph = event_graph_builder.compile()

initial_state = {
    "topic": "event streaming",
    "notes": [],
    "draft": "",
    "final": "",
}

event_graph.invoke(initial_state)

{'topic': 'event streaming',
 'notes': ['event streaming 需要观察节点更新',
  'event streaming 需要区分 updates 和 values'],
 'draft': '草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values',
 'final': '最终回答：草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values'}

## 4. stream_mode = updates

`updates` 只输出每个节点返回的增量更新。

适合前端展示：

```text
哪个节点刚刚产出了什么？
```

In [90]:
for chunk in event_graph.stream(initial_state, stream_mode="updates"):
    print(chunk)

{'collect_notes': {'notes': ['event streaming 需要观察节点更新', 'event streaming 需要区分 updates 和 values']}}
{'draft_answer': {'draft': '草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values'}}
{'finalize': {'final': '最终回答：草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values'}}


## 5. stream_mode = values

`values` 输出每一步后的完整 state。

适合调试：

```text
整个 state 现在长什么样？
```

In [91]:
for chunk in event_graph.stream(initial_state, stream_mode="values"):
    print(chunk)

{'topic': 'event streaming', 'notes': [], 'draft': '', 'final': ''}
{'topic': 'event streaming', 'notes': ['event streaming 需要观察节点更新', 'event streaming 需要区分 updates 和 values'], 'draft': '', 'final': ''}
{'topic': 'event streaming', 'notes': ['event streaming 需要观察节点更新', 'event streaming 需要区分 updates 和 values'], 'draft': '草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values', 'final': ''}
{'topic': 'event streaming', 'notes': ['event streaming 需要观察节点更新', 'event streaming 需要区分 updates 和 values'], 'draft': '草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values', 'final': '最终回答：草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values'}


## 6. stream_mode = debug

`debug` 会输出更细的任务事件，例如：

- task
- task_result

适合排查 graph 执行路径。

In [92]:
for index, event in enumerate(event_graph.stream(initial_state, stream_mode="debug")):
    print(
        "type=", event.get("type"),
        "name=", event.get("payload", {}).get("name"),
        "result=", event.get("payload", {}).get("result"),
    )
    if index >= 5:
        break

type= task name= collect_notes result= None
type= task_result name= collect_notes result= {'notes': ['event streaming 需要观察节点更新', 'event streaming 需要区分 updates 和 values']}
type= task name= draft_answer result= None
type= task_result name= draft_answer result= {'draft': '草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values'}
type= task name= finalize result= None
type= task_result name= finalize result= {'final': '最终回答：草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values'}


## 7. astream_events

当前环境没有同步 `stream_events`，但有异步 `astream_events`。

它会输出更接近 LangChain runnable event 的结构：

```text
event
name
run_id
metadata
data
```

下面用线程包装 `asyncio.run(...)`，这样在普通 Python 和 Notebook 环境里都更稳。

In [93]:
async def collect_astream_events() -> list[dict[str, Any]]:
    rows = []
    async for event in event_graph.astream_events(initial_state, version="v2"):
        rows.append(
            {
                "event": event.get("event"),
                "name": event.get("name"),
                "data_keys": sorted(event.get("data", {}).keys()),
            }
        )
        if len(rows) >= 8:
            break
    return rows


def run_async_in_thread(coro):
    box: dict[str, Any] = {}

    def runner():
        box["value"] = asyncio.run(coro)

    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()
    return box["value"]


for row in run_async_in_thread(collect_astream_events()):
    print(row)

{'event': 'on_chain_start', 'name': 'LangGraph', 'data_keys': ['input']}
{'event': 'on_chain_start', 'name': 'collect_notes', 'data_keys': ['input']}
{'event': 'on_chain_stream', 'name': 'collect_notes', 'data_keys': ['chunk']}
{'event': 'on_chain_end', 'name': 'collect_notes', 'data_keys': ['input', 'output']}
{'event': 'on_chain_stream', 'name': 'LangGraph', 'data_keys': ['chunk']}
{'event': 'on_chain_start', 'name': 'draft_answer', 'data_keys': ['input']}
{'event': 'on_chain_stream', 'name': 'draft_answer', 'data_keys': ['chunk']}
{'event': 'on_chain_end', 'name': 'draft_answer', 'data_keys': ['input', 'output']}


## 8. 检查当前环境是否支持 stream_events v3

官方新版示例常见写法是：

```python
stream = graph.stream_events(input, version="v3")
for message in stream.messages:
    print(message.text)
final_state = stream.output
```

但本环境的 compiled graph 暂时没有同步 `stream_events` 方法。教学上要先认识这个版本边界。

In [94]:
print("has stream:", hasattr(event_graph, "stream"))
print("has astream_events:", hasattr(event_graph, "astream_events"))
print("has stream_events:", hasattr(event_graph, "stream_events"))

has stream: True
has astream_events: True
has stream_events: True


## 9. 映射到 SSE

本仓库 Harness Chat Agent 使用 SSE 给前端返回事件。

LangGraph streaming 事件也可以映射成 SSE：

```text
event: node_update
data: {...}
```

下面只是格式演示，不启动 FastAPI 服务。

In [95]:
def to_sse(event_name: str, payload: dict) -> str:
    return "event: " + event_name + "\n" + "data: " + json.dumps(payload, ensure_ascii=False) + "\n"


for chunk in event_graph.stream(initial_state, stream_mode="updates"):
    print(to_sse("node_update", chunk))

event: node_update
data: {"collect_notes": {"notes": ["event streaming 需要观察节点更新", "event streaming 需要区分 updates 和 values"]}}

event: node_update
data: {"draft_answer": {"draft": "草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values"}}

event: node_update
data: {"finalize": {"final": "最终回答：草稿：event streaming 需要观察节点更新；event streaming 需要区分 updates 和 values"}}



## 10. stream mode 怎么选

| 方式 | 输出 | 适合用途 |
| --- | --- | --- |
| `updates` | 每个节点的增量更新 | 前端进度、节点结果 |
| `values` | 每一步后的完整 state | 调试 state 演化 |
| `debug` | 任务级调试事件 | 排查 graph 执行路径 |
| `astream_events(..., version="v2")` | runnable event 结构 | 更细粒度事件观察 |
| `stream_events(..., version="v3")` | typed projections | 生产应用同时消费 messages、values、interrupts、subgraphs 等投影 |

真实产品里通常会把内部事件转换成业务事件。

例如：

```text
collect_notes -> progress
draft_answer  -> answer_delta 或 draft_ready
finalize      -> final_answer
```

## 11. 和 Harness SSE 的关系

| LangGraph Streaming / Event Streaming | Harness SSE |
| --- | --- |
| node update | `tool_result` / `subagent_result` |
| full state values | ledger / run state |
| debug event | internal trace |
| typed projections / runnable events | model/tool lifecycle events |
| final state | `answer_delta` / final answer |

关键设计点：

```text
不要把所有内部事件原样暴露给前端。
应该把内部 graph event 转换成用户能理解的业务事件。
```

## 12. 本讲练习

请判断下面场景应该用哪种 stream mode：

1. 前端要显示“当前执行到哪个节点”。
2. 开发者要检查每一步完整 state。
3. 排查为什么某个节点没执行。
4. 想拿到 runnable lifecycle event。
5. 生产 UI 想同时消费 token、state、interrupt 和子图事件。

参考答案：

1. `updates`
2. `values`
3. `debug`
4. `astream_events`
5. 官方新版 `stream_events(..., version="v3")` 的 typed projections

## 13. 本讲小结

这一讲的核心：

```text
Event streaming 让 graph 执行过程可观察，而不是只等最终结果。
```

你现在应该能看懂：

- `stream_mode="updates"`
- `stream_mode="values"`
- `stream_mode="debug"`
- `astream_events(..., version="v2")`
- 官方新版 `stream_events(..., version="v3")` 的 typed projections
- 如何把 LangGraph event 映射成 SSE

下一讲可以继续学习 Memory。